<a href="https://colab.research.google.com/github/NihitaSoma/Microsoft-2B-document-intelligence-agent-harness/blob/chunking/phase_1_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

## Install libraries


In [1]:
import sys
import site
import os
import pathlib
import pandas as pd

# Ensure Windows user site-packages are visible to the kernel
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

# Set relative data path for Windows local workspace
data_path = os.path.join(pathlib.Path().resolve(), "data")

print(f"Data directory: {data_path}")
print(f"Files found: {len(os.listdir(data_path))}")

Data directory: C:\01_Data\Varsha\Break Through Tech AI\Microsoft-2B-document-intelligence-agent-harness\data
Files found: 11


In [1]:
# import os
# import pathlib
# import pandas as pd
# !pip install pymupdf
# !pip install pymupdf4llm
# !pip install python-docx
# !pip install 'markitdown[all]'
# !pip install -U langchain-text-splitters
# !pip install chromadb sentence-transformers

%pip install pymupdf pymupdf4llm python-docx "markitdown[all]" langchain-text-splitters chromadb sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ------------ --------------------------- 3.9/12.7 MB 18.3 MB/s eta 0:00:01
   ---------------------------- ----------- 8.9/12.7 MB 21.1 MB/s eta 0:00:01
   ---------------------------------------  12.6/12.7 MB 20.9 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 19.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ------------------ --------------------- 5.2/11.3 MB 26.4 MB/s eta 0:00:01
   ------------------------------------- -- 10.5/11.3 MB 24.6 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.3 MB 24.8 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 16.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   -----------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Data


### Import Files

In [2]:
data_path = './data'

In [3]:
dir_list = os.listdir(data_path)
print("Files and directories in '", data_path, "' :")
# prints all files
print(dir_list)

Files and directories in ' ./data ' :
['.gitkeep', 'northstar_academic_integrity_policy_2026.pdf', 'northstar_admissions_and_enrollment_guide_2026.pdf', 'northstar_attendance_and_course_participation_policy_2026.docx', 'northstar_examination_and_grading_policy_2025.pdf', 'northstar_examination_and_grading_policy_2026.pdf', 'northstar_internship_guidelines_2026.docx', 'northstar_laboratory_safety_handbook_2026.docx', 'northstar_library_services_and_use_rules_2026.pdf', 'northstar_scholarship_and_financial_aid_guide_2026.pdf', 'northstar_student_handbook_and_campus_services_2026.pdf']


### Sort Files

In [4]:
# Separate list of PDF's and DOCX's
all_pdf = []
all_docx = []

# Sort by PDF and DOCX
for file in dir_list:
  if file.endswith('.pdf'):
    all_pdf.append(file)
  elif file.endswith('.docx'):
    all_docx.append(file)

print(len(all_pdf), " PDFs: ", all_pdf)
print(len(all_docx), " DOCXs: ", all_docx)

7  PDFs:  ['northstar_academic_integrity_policy_2026.pdf', 'northstar_admissions_and_enrollment_guide_2026.pdf', 'northstar_examination_and_grading_policy_2025.pdf', 'northstar_examination_and_grading_policy_2026.pdf', 'northstar_library_services_and_use_rules_2026.pdf', 'northstar_scholarship_and_financial_aid_guide_2026.pdf', 'northstar_student_handbook_and_campus_services_2026.pdf']
3  DOCXs:  ['northstar_attendance_and_course_participation_policy_2026.docx', 'northstar_internship_guidelines_2026.docx', 'northstar_laboratory_safety_handbook_2026.docx']


# Phase 1: Extraction

In [5]:
# Pandas DF to store all extractable information
cols = ["filename", "filepath", "file_type", "md_text", "metadata"]
df = pd.DataFrame(columns=cols)

## PDF's - PyMuPDF



In [7]:
import pymupdf, pymupdf4llm

def extract_pdf_to_entry(pdf:str) -> list:
  """
  Extracts markdown content from pdf file.

  Args:
    pdf: local pdf file

  Returns
    data_entry: a row of information formatted as a list to add to pandas dataframe
  """

  # Get full file path
  file_path = os.path.join(data_path, pdf)

  # Generate markdown
  md_text = pymupdf4llm.to_markdown(file_path)

  # Get metadata
  metadata = pymupdf.open(file_path).metadata

  # Make data entry for this instance in order of: filename, filepath, file_type, md_text, metadata
  data_entry = [pdf, file_path, "pdf", md_text, metadata]

  return data_entry

# Add data_entry as a row in the df
for pdf in all_pdf:
  df.loc[len(df)] = extract_pdf_to_entry(pdf)

display(df.head(7))


,filename,filepath,file_type,md_text,metadata
0,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,**NORTHSTAR UNIVERSITY** | NSU-INTEGRITY-2026...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
1,northstar_admissions_and_enrollment_guide_2026...,./data\northstar_admissions_and_enrollment_gui...,pdf,**NORTHSTAR UNIVERSITY** | NSU-ADMIT-2026 \n\...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
2,northstar_examination_and_grading_policy_2025.pdf,./data\northstar_examination_and_grading_polic...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2025 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
3,northstar_examination_and_grading_policy_2026.pdf,./data\northstar_examination_and_grading_polic...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2026 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
4,northstar_library_services_and_use_rules_2026.pdf,./data\northstar_library_services_and_use_rule...,pdf,**NORTHSTAR UNIVERSITY** | NSU-LIB-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
5,northstar_scholarship_and_financial_aid_guide_...,./data\northstar_scholarship_and_financial_aid...,pdf,**NORTHSTAR UNIVERSITY** | NSU-AID-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
6,northstar_student_handbook_and_campus_services...,./data\northstar_student_handbook_and_campus_s...,pdf,**NORTHSTAR UNIVERSITY** | NSU-STUDENT-2026 \...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."


## DOCX's - python-docx + MarkItDown


In [8]:
from datetime import datetime
from docx import Document
from markitdown import MarkItDown


# Helper function to reformat python datetime objects to our standard format
def format_date(dt):
    if isinstance(dt, datetime):
        # Formats to a PDF-like date string: D:YYYYMMDDHHMMSS
        return f"D:{dt.strftime('%Y%m%d%H%M%S')}"
    return ""

# Iterate through all DOCX's - fill in DataFrame
def extract_docx_to_entry(docx:str) -> list:
  """
  Extracts markdown content from docx file.

  Args:
    docx: local docx file

  Returns
    data_entry: a row of information formatted as a list to add to pandas dataframe
  """
  # 0. Get full filepath
  file_path = os.path.join(data_path, docx)

  # 1. Get markdown text - MarkItDown
  md = MarkItDown()
  result = md.convert(file_path)
  md_text = result.text_content

  # 2.1. Get metadata - python-docx
  doc = Document(file_path) # Pass file_path here, not docx
  props = doc.core_properties

  # 2.2. Reformat metadata
  metadata = {
      "format": "DOCX",  # Explicit file format
      "title": props.title or "",
      "author": props.author or "",
      "subject": props.subject or "",
      "keywords": props.keywords or "",
      "creator": props.last_modified_by or "",  # Fallback to last modifier
      "producer": "python-docx & Microsoft Word",
      "creationDate": format_date(props.created),
      "modDate": format_date(props.modified),
      "trapped": "",
      "encryption": None,  # docx files aren't encrypted natively this way
  }

  # Make data entry for this instance in order of: filename, filepath, file_type, md_text, metadata
  data_entry = [docx, file_path, "docx", md_text, metadata]

  return data_entry

# Add data_entry as a row in the df
for docx in all_docx:
  df.loc[len(df)] = extract_docx_to_entry(docx)


display(df.tail(3))

,filename,filepath,file_type,md_text,metadata
14,northstar_attendance_and_course_participation_...,./data\northstar_attendance_and_course_partici...,docx,**Attendance and Course Participation Policy**...,"{'format': 'DOCX', 'title': 'Attendance and Co..."
15,northstar_internship_guidelines_2026.docx,./data\northstar_internship_guidelines_2026.docx,docx,**Undergraduate Internship Guidelines**\n\n**A...,"{'format': 'DOCX', 'title': 'Undergraduate Int..."
16,northstar_laboratory_safety_handbook_2026.docx,./data\northstar_laboratory_safety_handbook_20...,docx,**Undergraduate Laboratory Safety Handbook**\n...,"{'format': 'DOCX', 'title': 'Undergraduate Lab..."


In [9]:
# Examine current Dataframe
display(df.head(10))

,filename,filepath,file_type,md_text,metadata
0,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,**NORTHSTAR UNIVERSITY** | NSU-INTEGRITY-2026...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
1,northstar_admissions_and_enrollment_guide_2026...,./data\northstar_admissions_and_enrollment_gui...,pdf,**NORTHSTAR UNIVERSITY** | NSU-ADMIT-2026 \n\...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
2,northstar_examination_and_grading_policy_2025.pdf,./data\northstar_examination_and_grading_polic...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2025 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
3,northstar_examination_and_grading_policy_2026.pdf,./data\northstar_examination_and_grading_polic...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2026 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
4,northstar_library_services_and_use_rules_2026.pdf,./data\northstar_library_services_and_use_rule...,pdf,**NORTHSTAR UNIVERSITY** | NSU-LIB-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
5,northstar_scholarship_and_financial_aid_guide_...,./data\northstar_scholarship_and_financial_aid...,pdf,**NORTHSTAR UNIVERSITY** | NSU-AID-2026 \n\n#...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
6,northstar_student_handbook_and_campus_services...,./data\northstar_student_handbook_and_campus_s...,pdf,**NORTHSTAR UNIVERSITY** | NSU-STUDENT-2026 \...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
7,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,**NORTHSTAR UNIVERSITY** | NSU-INTEGRITY-2026...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
8,northstar_admissions_and_enrollment_guide_2026...,./data\northstar_admissions_and_enrollment_gui...,pdf,**NORTHSTAR UNIVERSITY** | NSU-ADMIT-2026 \n\...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
9,northstar_examination_and_grading_policy_2025.pdf,./data\northstar_examination_and_grading_polic...,pdf,**NORTHSTAR UNIVERSITY** | NSU-EXAM-2025 \n\n...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."


# Phase 2: Chunk Text

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter
# chunk_size = maximum number of characters in each chunk
# chunk_overlap = number of characters shared between neighboring chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=550,
    chunk_overlap=75,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Store all chunks here
chunked_data = []

# Go through every file in the DataFrame
for _, row in df.iterrows():

    # Get the extracted markdown text
    text = row["md_text"]

    # Skip files with no extracted text
    if not text or not text.strip():
        continue

    # Split the document into chunks
    chunks = text_splitter.split_text(text)

    # Create one entry for every chunk
    for chunk_id, chunk in enumerate(chunks):

        chunked_data.append({
            "filename": row["filename"],
            "filepath": row["filepath"],
            "file_type": row["file_type"],
            "chunk_id": chunk_id,
            "text": chunk,
            "metadata": row["metadata"]
        })

# Create a new DataFrame containing the chunks
chunks_df = pd.DataFrame(chunked_data)

# Examine the results
display(chunks_df.head(10))

C:\Users\Varsha\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,filename,filepath,file_type,chunk_id,text,metadata
0,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,0,**NORTHSTAR UNIVERSITY** | NSU-INTEGRITY-2026...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
1,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,1,## **1. Prohibited Conduct** \n\nAcademic misc...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
2,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,2,Reusing one's own prior submitted work without...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
3,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,3,## **2. Reporting and Initial Review** \n\nAn ...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
4,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,4,The instructor may assign a temporary grade of...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
5,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,5,## **3. Findings and Sanctions** \n\nThe stand...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
6,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,6,|**Violation context**|**Possible response**|\...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
7,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,7,## **4. Appeals and Records** \n\nA student ma...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
8,northstar_academic_integrity_policy_2026.pdf,./data\northstar_academic_integrity_policy_202...,pdf,8,- Appeal deadline: 10 calendar days. \n\n- Nor...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."
9,northstar_admissions_and_enrollment_guide_2026...,./data\northstar_admissions_and_enrollment_gui...,pdf,0,**NORTHSTAR UNIVERSITY** | NSU-ADMIT-2026 \n\...,"{'format': 'PDF 1.7', 'title': '', 'author': '..."


# Phase 3: Embeddings

In [18]:
# Drop duplicate chunk records from chunks_df
chunks_df = chunks_df.drop_duplicates(subset=["filename", "chunk_id"]).reset_index(drop=True)

print(f"Total unique chunks to index: {len(chunks_df)}")

Total unique chunks to index: 88


In [19]:
import os
import chromadb
import torch
from sentence_transformers import SentenceTransformer

# 1. Initialize Persistent ChromaDB Client
persist_dir = "./chroma_db"
os.makedirs(persist_dir, exist_ok=True)
client = chromadb.PersistentClient(path=persist_dir)

collection = client.get_or_create_collection(
    name="northstar_docs",
    metadata={"hnsw:space": "cosine"}
)

# 2. Load Nomic Embeddings Model
print("Loading nomic-embed-text-v1.5...")
embed_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

# --- PATCH FOR TRANSFORMERS COMPATIBILITY ON PYTHON 3.13 ---
# In modern transformers, get_extended_attention_mask was moved/removed from some modules.
# We inject a compatible implementation directly onto the underlying model object.
def _patch_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if device is None:
        device = attention_mask.device
    if dtype is None:
        dtype = self.dtype if hasattr(self, "dtype") else torch.float32

    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        extended_attention_mask = attention_mask[:, None, None, :]
    else:
        raise ValueError(f"Wrong shape for attention_mask (shape {attention_mask.shape})")

    extended_attention_mask = extended_attention_mask.to(dtype=dtype)
    extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
    return extended_attention_mask

# Dynamically patch the core module
core_model = embed_model[0].auto_model
if not hasattr(core_model, "get_extended_attention_mask"):
    core_model.get_extended_attention_mask = _patch_extended_attention_mask.__get__(core_model, type(core_model))
# -----------------------------------------------------------

# 3. Prepare Documents, Metadata, and IDs
documents = []
metadatas = []
ids = []

for _, row in chunks_df.iterrows():
    chunk_uid = f"{row['filename']}#chunk_{row['chunk_id']}"
    ids.append(chunk_uid)
    documents.append(row["text"])
    
    doc_meta = row["metadata"] if isinstance(row["metadata"], dict) else {}
    clean_meta = {
        "filename": str(row["filename"]),
        "filepath": str(row["filepath"]),
        "file_type": str(row["file_type"]),
        "chunk_id": int(row["chunk_id"]),
        "doc_title": str(doc_meta.get("title", "") or ""),
        "doc_author": str(doc_meta.get("author", "") or ""),
        "doc_format": str(doc_meta.get("format", "") or row["file_type"]),
        "creation_date": str(doc_meta.get("creationDate", "") or ""),
        "mod_date": str(doc_meta.get("modDate", "") or ""),
    }
    metadatas.append(clean_meta)

# 4. Generate Embeddings (Nomic document task prefix)
print(f"Encoding {len(documents)} chunks...")
nomic_prefixed_texts = [f"search_document: {doc}" for doc in documents]
embeddings = embed_model.encode(nomic_prefixed_texts, show_progress_bar=True, batch_size=32).tolist()

# 5. Upsert to ChromaDB
print("Upserting records into ChromaDB...")
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Successfully stored and persisted {collection.count()} chunks in ChromaDB!")

Loading nomic-embed-text-v1.5...


<All keys matched successfully>


Encoding 88 chunks...


Batches: 100%|██████████| 3/3 [00:28<00:00,  9.48s/it]


Upserting records into ChromaDB...
Successfully stored and persisted 88 chunks in ChromaDB!


# Phase 4: Depluplication & Verification

Replace the Phase 3 position IDs (`<filename>#chunk_<n>`) with content-hash IDs and remove the old records.

In [ ]:
from chunk_index import assign_chunk_ids, sync_collection

needed = ["chunks_df", "documents", "metadatas", "embeddings", "embed_model", "collection"]
missing = [name for name in needed if name not in globals()]
if missing:
    print(f"Missing: {', '.join(missing)}. Run Phase 3 (the embeddings cell) first.")
else:
    assert len(documents) == len(metadatas) == len(embeddings) == len(chunks_df), "Phase 3 lists do not match chunks_df. Re-run Phase 3."
    chunk_ids = assign_chunk_ids(chunks_df["filename"].tolist(), documents)
    assert len(set(chunk_ids)) == len(chunk_ids), "Chunk IDs are not unique."
    print(f"Built {len(chunk_ids)} unique IDs, e.g. {chunk_ids[0]}")

    first_sync = sync_collection(collection, chunk_ids, documents, metadatas, embeddings)
    print(first_sync)

Check that the collection holds exactly one record per chunk and no old `#chunk_` IDs.

In [ ]:
def report(label, ok):
    print(f"{'PASS' if ok else 'FAIL'}: {label}")

old_ids = [record_id for record_id in collection.get(include=[])["ids"] if "#chunk_" in record_id]
report(f"collection.count() == len(chunks_df) ({collection.count()} vs {len(chunks_df)})", collection.count() == len(chunks_df))
report(f'no "#chunk_" IDs left ({len(old_ids)} found)', not old_ids)

Sync again: the count must not change, because re-indexing updates records instead of adding new ones.

In [ ]:
count_before = collection.count()
second_sync = sync_collection(collection, chunk_ids, documents, metadatas, embeddings)
print(second_sync)
report(f"count unchanged after second sync ({count_before} -> {collection.count()})", collection.count() == count_before)

Retrieval spot check: the expected file should be in the top 3 results for each question.

In [ ]:
EXAM_2025 = "northstar_examination_and_grading_policy_2025.pdf"
EXAM_2026 = "northstar_examination_and_grading_policy_2026.pdf"

def search(question, n_results=3):
    """Embed a question with the Nomic query prefix and return (filename, distance) pairs, closest first."""
    query_embedding = embed_model.encode([f"search_query: {question}"]).tolist()
    result = collection.query(query_embeddings=query_embedding, n_results=n_results, include=["metadatas", "distances"])
    return [(meta["filename"], distance) for meta, distance in zip(result["metadatas"][0], result["distances"][0])]

sample_questions = [
    ("What is the deadline to request rescheduling a final exam?", EXAM_2026),
    ("How long can I borrow a standard library book?", "northstar_library_services_and_use_rules_2026.pdf"),
    ("What tuition refund do I get if I withdraw in the third week?", "northstar_student_handbook_and_campus_services_2026.pdf"),
    ("What grade point average (GPA) is required to renew the Northstar Achievement Scholarship?", "northstar_scholarship_and_financial_aid_guide_2026.pdf"),
    ("How many supervised hours does a three-credit internship need?", "northstar_internship_guidelines_2026.docx"),
]

for question, expected_file in sample_questions:
    top3 = search(question)
    print(question)
    for rank, (filename, distance) in enumerate(top3, start=1):
        print(f"  {rank}. {filename}  distance={distance:.4f}")
    report(f"{expected_file} in top 3", expected_file in [filename for filename, _ in top3])

    if expected_file == EXAM_2026:
        # The 2025 policy is superseded; show how it ranks against the 2026 one.
        ranked = search(question, n_results=collection.count())
        rank_2025 = next((rank for rank, (filename, _) in enumerate(ranked, start=1) if filename == EXAM_2025), None)
        print(f"  Superseded {EXAM_2025} first appears at rank {rank_2025} of {len(ranked)}")
    print()

Question that is not in the documents: distances only, to compare with the answerable questions above.

In [ ]:
off_topic_question = "What is the campus parking policy?"
print(off_topic_question)
for rank, (_, distance) in enumerate(search(off_topic_question), start=1):
    print(f"  {rank}. distance={distance:.4f}")